In [1]:
#Download dataset from 
#https://www.kaggle.com/datasets/omkargurav/face-mask-dataset?resource=download

In [2]:
#Import libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

In [3]:
#Defining path
train_dir = 'data/'

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

#This code creates an ImageDataGenerator object in TensorFlow/Keras 
#for image preprocessing and data augmentation.
# | Parameter              | Description                                                                  |
# | ---------------------- | ---------------------------------------------------------------------------- |
# | `rescale=1./255`       | Converts pixel values from 0–255 to 0–1. Helps neural networks train better. |
# | `validation_split=0.2` | Reserves 20% of images for validation and 80% for training.                  |
# | `rotation_range=20`    | Randomly rotates images up to ±20 degrees.                                   |
# | `zoom_range=0.2`       | Randomly zooms images in/out by 20%.                                         |
# | `horizontal_flip=True` | Randomly flips images horizontally.                                          |

In [5]:
#Traing data object - it takes 80% of images
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

#Validation data object - it takes 20% of images
val_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 6043 images belonging to 2 classes.
Found 1510 images belonging to 2 classes.


In [8]:
# # total = 6043 + 1510
# # total
# total = 3725 + 3828
# print(total)

In [13]:
#Build NN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
model = Sequential([
    Conv2D(128, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(16, activation = 'relu'),
    Dropout(0.5),
    Dense(1, activation = 'sigmoid')
])

In [14]:
#Model compile
model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])

In [15]:
#Training
history = model.fit(train_data, epochs = 10, validation_data = val_data)

Epoch 1/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.7288 - loss: 0.4868 - val_accuracy: 0.8861 - val_loss: 0.3122
Epoch 2/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 118s 624ms/step - accuracy: 0.7961 - loss: 0.3967 - val_accuracy: 0.9113 - val_loss: 0.2344
Epoch 3/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 110s 579ms/step - accuracy: 0.8223 - loss: 0.3563 - val_accuracy: 0.9166 - val_loss: 0.2366
Epoch 4/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 106s 562ms/step - accuracy: 0.8357 - loss: 0.3236 - val_accuracy: 0.9113 - val_loss: 0.2203
Epoch 5/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 99s 523ms/step - accuracy: 0.8545 - loss: 0.3242 - val_accuracy: 0.9046 - val_loss: 0.2350
Epoch 6/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 116s 614ms/step - accuracy: 0.8785 - loss: 0.2978 - val_accuracy: 0.9232 - val_loss: 0.2021
Epoch 7/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 116s 614ms/step - accuracy: 0.9085 - loss: 0.2626 - val_accuracy: 0.9185 - val_loss: 0.2070
Epoch 8/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 118s 625ms/step - accuracy: 0.9082 - los

In [20]:
from tensorflow.keras.preprocessing import image
imagename = 'image2.jpg'
img = image.load_img(imagename, target_size=(150, 150))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)
result = model.predict(img_array)
if result[0,0] < 0.51:
    print("Person is with mask")
else:
    print("Person is without mask")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Person is without mask


In [21]:
model.save("face-mask-detector.keras")